In [1]:
import pandas as pd
import numpy as np

# Load the saved data from Phase 1
df = pd.read_csv('../data/raw_loaded.csv')
print(f"Shape: {df.shape}")
print("Data loaded successfully")

Shape: (101766, 51)
Data loaded successfully


In [2]:
# Drop columns that are too empty to be useful
df = df.drop(columns=['weight', 'payer_code'])

# Replace all '?' with 'Unknown'
df = df.replace('?', 'Unknown')

print("=== Columns dropped and missing values fixed ===")
print(f"New shape: {df.shape}")

# Verify no more '?' exist
total_q = (df == '?').sum().sum()
print(f"Remaining '?' values: {total_q}")

=== Columns dropped and missing values fixed ===
New shape: (101766, 49)
Remaining '?' values: 0


In [3]:
# Remove duplicate patients - keep only first visit per patient
df = df.drop_duplicates(subset='patient_nbr', keep='first')

print(f"Shape after removing duplicates: {df.shape}")
print(f"Patients removed: {101766 - len(df)}")

Shape after removing duplicates: (71518, 49)
Patients removed: 30248


In [4]:
# Remove rows where discharge disposition is death or hospice
# These patients can't be readmitted - they skew the model
df = df[~df['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])]

print(f"Shape after removing death/hospice: {df.shape}")
print(f"Rows removed: {71518 - len(df)}")

Shape after removing death/hospice: (69973, 49)
Rows removed: 1545


In [5]:
# Fix age column - it comes as ranges like '[70-80)' 
# Convert to numeric midpoint
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25,
    '[30-40)': 35, '[40-50)': 45, '[50-60)': 55,
    '[60-70)': 65, '[70-80)': 75, '[80-90)': 85,
    '[90-100)': 95
}

df['age'] = df['age'].map(age_map)

print("=== Age column fixed ===")
print(df['age'].value_counts().sort_index())

=== Age column fixed ===
age
5       153
15      534
25     1121
35     2692
45     6828
55    12349
65    15684
75    17750
85    11102
95     1760
Name: count, dtype: int64


In [6]:
# Save the cleaned dataset
df.to_csv('../data/cleaned_data.csv', index=False)

print("=== Cleaning Summary ===")
print(f"Original rows: 101,766")
print(f"Final rows: {len(df)}")
print(f"Rows removed: {101766 - len(df)}")
print(f"Columns: {df.shape[1]}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nReadmission rate: {df['target'].mean()*100:.1f}%")
print("\nSaved to data/cleaned_data.csv")

=== Cleaning Summary ===
Original rows: 101,766
Final rows: 69973
Rows removed: 31793
Columns: 49

Target distribution:
target
0    63696
1     6277
Name: count, dtype: int64

Readmission rate: 9.0%

Saved to data/cleaned_data.csv
